# QGECAF quality-stratified performance and gate-behaviour analysis

This notebook analyses the ten completed QGECAF outer folds without retraining. Fold-specific quality thresholds are calculated only from each saved inner-training partition and then applied to that fold's outer holdout.

Place this notebook and `QGECAF_Quality_Stratified_Gate_Analysis.py` in the same folder, select the `chaitalienv` kernel, and run the cells in order.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

from QGECAF_Quality_Stratified_Gate_Analysis import Config, run

In [ ]:
cfg = Config(
    qgecaf_root=(
        r'D:\chaitali\qgacef'
        r'\QGECAF_REVIEWER_REVISION_NESTED_BGWO'
    ),
    output=(
        r'D:\chaitali\qgacef'
        r'\QGECAF_REVIEWER_REVISION_NESTED_BGWO'
        r'\quality_stratified_gate_analysis'
    ),
    shuffled_gate_root='',  # add the future matched G3 result path here
    lower_percentile=33.0,
    upper_percentile=67.0,
    ece_bins=15,
    bootstrap_repetitions=2000,
    seed=42,
    overwrite=False,
)
cfg

In [ ]:
root = Path(cfg.qgecaf_root)
required = [
    root / 'locked_patient_fold_assignments.csv',
    root / 'raw_features_presplit.npz',
]
for fold in range(1, 11):
    fold_dir = root / 'cross_validation' / f'fold_{fold:02d}'
    required.extend([
        fold_dir / 'patient_partition_audit.csv',
        fold_dir / 'preprocessing.joblib',
        fold_dir / 'outer_holdout_predictions.csv',
    ])
missing = [path for path in required if not path.is_file()]
print('Required files:', len(required))
print('Missing files:', len(missing))
for path in missing[:20]:
    print('MISSING:', path)
assert not missing, 'Resolve the missing files before running the analysis.'
print('PASS: All ten fold audits, preprocessors and prediction files exist.')

## Run the analysis

This performs no model fitting. Most execution time is spent on the patient-clustered bootstrap confidence intervals.

In [ ]:
run(cfg)

In [ ]:
result_root = Path(cfg.output)
display(pd.read_csv(result_root / 'quality_thresholds_by_fold.csv'))
display(pd.read_csv(result_root / 'quality_stratified_fold_metrics.csv'))
display(pd.read_csv(result_root / 'quality_stratified_10fold_summary.csv'))
display(pd.read_csv(result_root / 'quality_group_friedman_tests.csv'))
display(pd.read_csv(result_root / 'quality_group_paired_wilcoxon_holm.csv'))
print((result_root / 'automated_interpretation.txt').read_text(encoding='utf-8'))
print('Run complete:', (result_root / 'RUN_COMPLETE.json').is_file())

In [ ]:
display(Image(filename=str(result_root / 'quality_gate_distribution_boxplot.png')))
display(Image(filename=str(result_root / 'quality_stratified_metric_panels.png')))
display(Image(filename=str(result_root / 'quality_score_vs_gate_lambda.png')))